In [0]:
%sql
USE CATALOG databricks_proyecto_jhon;
-- Creamos el schema
CREATE SCHEMA IF NOT EXISTS lakehouse;
-- Creamos el volumen
CREATE VOLUME IF NOT EXISTS lakehouse.datos_crudos;

In [0]:
from pyspark.sql import functions as F

# Guardamos en una variable la ruta exacta de tu archivo crudo. El protocolo abfss:// es el estándar de seguridad nativo de Azure para leer datos directamente desde tu Azure Data Lake Storage Gen2.
RUTA_LANDING_BCRP = "abfss://lakehouse@datalakejhon2026.dfs.core.windows.net/datos_crudos/landing/tipo_cambio_bcrp.json"

# Lee el archivo desde el Data Lake y lo convierte en un DataFrame (una tabla en memoria). La instrucción option("multiline", "true") es crítica aquí: por defecto, Spark espera que los JSON tengan un registro por línea. Como las APIs suelen entregar JSONs formateados con saltos de línea y anidaciones, esta opción le permite a Spark leer la estructura completa sin romperse.

df_crudo_bcrp = spark.read.option("multiline", "true").json(RUTA_LANDING_BCRP)

# Toma los datos crudos y les agrega dos nuevas columnas de metadatos usando el comando .withColumn(): 
# _fecha_carga: Utiliza F.current_timestamp() para estampar la fecha y hora exactas en las que el servidor procesó el archivo.
# _origen: Utiliza F.lit() para inyectar un valor "literal" o fijo. En este caso, escribe "API_BCRP" en todas las filas para que, si en el futuro mezclas múltiples fuentes, sepas exactamente de dónde vino esta fila.

df_bronce_bcrp = (df_crudo_bcrp
    .withColumn("_fecha_carga", F.current_timestamp())
    .withColumn("_origen", F.lit("API_BCRP"))
)

# Guardado en formato Delta (Capa Bronce)
#.format("delta"): Guarda los datos usando la tecnología Delta Lake, garantizando transacciones seguras (ACID) y compresión avanzada.
#.mode("overwrite"): Le indica que si la tabla ya existe de una ejecución anterior, la reemplace por completo con estos nuevos datos.
#.option("overwriteSchema", "true"): Activa la evolución de esquema. Si mañana el BCRP agrega una nueva columna al JSON, Spark no fallará; simplemente actualizará la estructura de tu tabla automáticamente.
#.saveAsTable(...): No solo guarda los archivos físicos, sino que registra oficialmente la tabla en tu Unity Catalog (catálogo.esquema.tabla).

(df_bronce_bcrp.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") 
    .saveAsTable("databricks_proyecto_jhon.lakehouse.bronce_tipo_cambio_bcrp")
)

print(f" BCRP guardado en Bronce: {df_bronce_bcrp.count()} registros.")

display(df_bronce_bcrp)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F

# 1. Ruta donde el generador guardó los miles de registros
RUTA_LANDING_VENTAS = "/Volumes/databricks_proyecto_jhon/lakehouse/datos_crudos/landing/ventas_historico.json"

# 2. Todo como texto para evitar que datos corruptos rompan la ingesta
ESQUEMA = StructType([StructField(c, StringType(), True) for c in [
    "id_venta", "fecha", "id_cliente", "nombre_cliente", "canal", "ciudad",
    "id_producto", "nombre_producto", "categoria", "marca",
    "cantidad", "precio_unit_pen", "costo_unit_usd", "estado"
]])

# 3. Lectura de los casi 200 mil registros crudos
df_crudo_ventas = (spark.read
    .option("multiline", "true")
    .schema(ESQUEMA)
    .json(RUTA_LANDING_VENTAS))

# 4. Enriquecimiento de auditoría
df_bronce_ventas = (df_crudo_ventas
    .withColumn("_fecha_carga", F.current_timestamp())
    .withColumn("_origen", F.lit("GENERADOR_VOLUMEN")))

# 5. Guardado en formato Delta
(df_bronce_ventas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("databricks_proyecto_jhon.lakehouse.bronce_ventas"))

print(f"✅ Volumen cargado en Bronce: {df_bronce_ventas.count():,} registros.")